#  **Практическое занятие №4. Метод градиентного спуска.**

In [ ]:
import numpy as np

import matplotlib.pyplot as plt

%matplotlib inline

## Простые примеры

### Одномерный случай


Функция:
$y = f(x) = x^2$

Производная: $f'(x)= 2x$

Начальное приближение: $x_0 =1.6$

Размер шага: $\gamma = 0.2$

Шаг градиентного спуска:
$x_{i+1} = x_i - \gamma 2x_i$


In [ ]:
def func(x):
    return x ** 2

def grad(x):
    return 2 * x

def step(x, lr):
    return x - lr * grad(x)

In [ ]:
lr = 0.2
x_hist = [1.6]

for i in range(20):
    x_curr = x_hist[-1]
    x_next = step(x_curr, lr)
    x_hist.append(x_next)

# YOUR CODE HERE
x_hist = np.array(x_hist)
y_hist = func(x_hist)

In [ ]:
x_hist

In [ ]:
x_i+1 - x_i

In [ ]:
plt.figure(figsize=(16,4))


plt.subplot(1, 3, 1)
plt.title('GD trajectory')

x_grid = np.arange(-2, 2.1, 0.1)
y_grid = func(x_grid)
plt.plot(x_grid, y_grid)

plt.scatter(x_hist, y_hist, c='r', zorder=10)

plt.grid()


plt.subplot(1, 3, 2)
plt.title('GD param diff')

diff = np.abs(np.diff(x_hist))
plt.plot(np.arange(len(diff)), diff)

plt.grid()


plt.subplot(1, 3, 3)
plt.title('GD func')

plt.plot(np.arange(len(y_hist)), y_hist)

plt.grid()


plt.show()

### Двумерный случай


Функция:
$z = f(x_1, x_2)\ =x_1^2 + x_2^2$


Градиент: $\nabla f(x_1, x_2) = (2x_1, 2x_2)$

Начальное приближение: $(x_0, y_0) = (0.8, 0.8)$

Размер шага: $\gamma = 0.1$

Шаг градиентного спуска:
$(x^{i+1}, y^{i+1}) = (x^i, y^i) - \gamma(2x^i, 2y^i)$

In [ ]:
def func(x):
    return np.sum(x ** 2)

def grad(x):
    return x * 2

def step(x, lr):
    return x - lr * grad(x)

In [ ]:
lr = 0.1
x_hist = [np.array([0.8, 0.8])]

for i in range(20):
    x_curr = x_hist[-1]
    x_next = step(x_curr, lr)
    x_hist.append(x_next)

# YOUR CODE HERE
x_hist = np.array(x_hist)
y_hist = [func(x) for x in x_hist]

In [ ]:
def make_levels(grid, func, num_levels=50):
    X, Y = np.meshgrid(grid, grid)
    Z = np.empty_like(X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            Z[i, j] = func(np.array([X[i, j], Y[i, j]]))
    levels = np.geomspace(np.min(Z), np.max(Z), num=num_levels)
    return X, Y, Z, levels

In [ ]:
plt.figure(figsize=(16,4))


plt.subplot(1, 10, (1, 4))
plt.title('GD trajectory')

plt.contourf(*make_levels(np.linspace(-1, 1, 100), func))
plt.colorbar()

plt.scatter(x_hist[:,0], x_hist[:,1], c='r', s=10, zorder=10)


plt.subplot(1, 10, (5, 7))
plt.title('GD param diff')

diff = np.linalg.norm(x_hist[1:] - x_hist[:-1], axis=1)
plt.plot(np.arange(len(diff)), diff)

plt.grid()


plt.subplot(1, 10, (8, 10))
plt.title('GD func')

plt.plot(np.arange(len(y_hist)), y_hist)

plt.grid()


plt.subplots_adjust(wspace=1)
plt.show()

## Линейная регрессия

### Вспомним лекцию

Модель:
$$
f(X) = X w
$$

Лосс:
$$
L(w, X, y) = MSE(y, X w) =  \|y - X w\|^2_2 = \sum_i (y_i - \langle X_i, w \rangle)^2
$$

Градиент:
$$
\nabla L(w) = 2X^T(X w - y)
$$

Шаг градиентного спуска:
$$
w^{i+1} = w^i - \gamma \nabla L(w^i)
$$

**Задача:** доказать, что формула градиента верна.

Действительно, пусть $v = 2 X^T (X w - y)$, тогда:

$$
\nabla L_j(w) = \sum_i 2 (y_i - \langle X_i, w \rangle) (- X_{i,j}) = 2 \sum_i X_{i,j} (\langle X_i, w \rangle - y_i)
$$

$$
v_j = 2 \langle X_{*,j}, Xw - y \rangle = 2 \sum_i X_{i,j} (\langle X_i, w \rangle - y_i) = \nabla L_j(w)
$$

### Сгенерируем датасет

In [ ]:
n_features = 2
n_objects = 500

np.random.seed(10)

X = np.random.uniform(-10, 10, (n_objects, n_features))

w = np.random.randn(n_features)

y = X @ w + np.random.randn(n_objects)

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(X[:,0], X[:,1], c=y)
plt.colorbar()

plt.show()

### Реализуем градиентный спуск

In [ ]:
X

In [ ]:
w - размерность 2
X - размерность 100 * 2
y - размерность 100

In [ ]:
def loss(w, X, y):
    return np.sum((X @ w - y) ** 2)
    
def grad(w, X, y):
    return 2 * X.T @ (X @ w - y)

def step(w, X, y, lr):
    return w - lr * grad(w, X, y)

In [ ]:
lr = 1e-5
w_hist = [np.zeros(n_features)]
l_hist = [loss(w_hist[-1], X, y)]

for i in range(20):
    w_hist.append(step(w_hist[-1], X, y, lr))
    l_hist.append(loss(w_hist[-1], X, y))

w_hist = np.array(w_hist)

In [ ]:
[ 0.3572789 , -0.18746801]
5000000, 2

100, 2

In [ ]:
w_hist

In [ ]:
w

In [ ]:
plt.figure(figsize=(16,4))


plt.subplot(1, 10, (1, 4))
plt.title('GD trajectory')

plt.contourf(*make_levels(np.linspace(-2, 2), lambda w: loss(w, X, y)))
plt.colorbar()

plt.scatter(w_hist[:,0], w_hist[:,1], c='r', s=10, zorder=10)


plt.subplot(1, 10, (5, 7))
plt.title('GD param diff')

diff = np.linalg.norm(w_hist[1:] - w_hist[:-1], axis=1)
plt.plot(np.arange(len(diff)), diff)

plt.grid()


plt.subplot(1, 10, (8, 10))
plt.title('GD loss')

plt.plot(np.arange(len(l_hist)), l_hist)

plt.grid()

plt.subplots_adjust(wspace=1)
plt.show()

### Добавим динамический learning_rate

In [ ]:
lr = 1e-4

w_hist = [np.zeros(n_features)]
l_hist = [loss(w_hist[-1], X, y)]

for i in range(20):
    w_hist.append(step(w_hist[-1], X, y, lr))
    l_hist.append(loss(w_hist[-1], X, y))
    # update lr
    if l_hist[-1] > l_hist[-2]:
        lr = lr / 10
    print(lr)

w_hist = np.array(w_hist)

In [ ]:
plt.figure(figsize=(16,4))


plt.subplot(1, 10, (1, 4))
plt.title('GD trajectory')

plt.contourf(*make_levels(np.linspace(-5, 5), lambda w: loss(w, X, y)))
plt.colorbar()

plt.scatter(w_hist[:,0], w_hist[:,1], c='r', s=10, zorder=10)


plt.subplot(1, 10, (5, 7))
plt.title('GD param diff')

diff = np.linalg.norm(w_hist[1:] - w_hist[:-1], axis=1)
plt.plot(np.arange(len(diff)), diff)

plt.grid()


plt.subplot(1, 10, (8, 10))
plt.title('GD loss')

plt.plot(np.arange(len(l_hist)), l_hist)

plt.grid()

plt.subplots_adjust(wspace=1)
plt.show()

### Реализуем SGD

In [ ]:
lr = 5 * 1e-4
chunk_size = 100

w_hist = [np.zeros(n_features)]
l_hist = [loss(w_hist[-1], X, y)]

for i in range(20):
    ind = np.random.randint(0, high=len(X), size=chunk_size)
    print(ind)
    X_chunk = X[ind,:]
    y_chunk = y[ind]
    w_hist.append(step(w_hist[-1], X_chunk, y_chunk, lr))
    l_hist.append(loss(w_hist[-1], X_chunk, y_chunk))

w_hist = np.array(w_hist)

In [ ]:
plt.figure(figsize=(16,4))


plt.subplot(1, 10, (1, 4))
plt.title('GD trajectory')

plt.contourf(*make_levels(np.linspace(-2, 2), lambda w: loss(w, X, y)))
plt.colorbar()

plt.scatter(w_hist[:,0], w_hist[:,1], c='r', s=10, zorder=10)


plt.subplot(1, 10, (5, 7))
plt.title('GD param diff')

diff = np.linalg.norm(w_hist[1:] - w_hist[:-1], axis=1)
plt.plot(np.arange(len(diff)), diff)

plt.grid()


plt.subplot(1, 10, (8, 10))
plt.title('GD loss')

plt.plot(np.arange(len(l_hist)), l_hist)

plt.grid()

plt.subplots_adjust(wspace=1)
plt.show()

In [ ]:
l_hist